# 02 — Relative Valuation

This notebook moves from data preparation to valuation analysis. The processed dataset created in notebook 01 is now used to build an initial trading comparables framework for Grupo Mateus (`GMAT3.SA`).

The goal is to estimate an implied valuation range for GMAT3 using peer EV/EBITDA multiples. This is not yet an investment recommendation, nor is it a final target price.

Relative valuation depends on peer comparability and analyst judgment. The numbers provide structure, but the interpretation must account for business quality, leverage, growth, earnings normalization, and whether each peer is truly comparable.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


## 1. Load Processed Dataset

The notebook starts by loading the processed dataset created in notebook 01. This dataset already combines raw financial fundamentals, market data, net debt, enterprise value, annualized EBITDA, and the initial EV/EBITDA calculation.


In [2]:
processed_data_path = Path("../data/processed/master_valuation_dataset.csv")
valuation_df = pd.read_csv(processed_data_path)

valuation_df


,ticker,company,sector,subsector,period,report_date,currency,revenue,ebitda,ebit,net_income,cash,total_debt,shares_outstanding,valuation_date,current_price,market_cap,market_cap_mn,net_debt,enterprise_value,ebitda_annualized,ev_ebitda_annualized
0,GMAT3.SA,Grupo Mateus,Consumer Staples,Food Retail,1T26,2026-03-31,BRL,9402,400,NaN,213,1984,2720,"2,300,047,621.00",2026-05-28,4.26,9798203392,"9,798.20",736,"10,534.20",1600,6.58
1,ASAI3.SA,Assai,Consumer Staples,Cash & Carry,1T26,2026-03-31,BRL,20600,1000,NaN,174,4366,16374,"1,353,531,000.00",2026-05-28,9.25,12411186176,"12,411.19",12008,"24,419.19",4000,6.10
2,PCAR3.SA,GPA,Consumer Staples,Food Retail,1T26,2026-03-31,BRL,4374,458,NaN,-1347,943,4173,NaN,2026-05-28,1.99,978954240,978.95,3230,"4,208.95",1832,2.30


## 2. Separate Target and Peers

GMAT3 is the target company. ASAI3 and PCAR3 are the comparable companies used to frame GMAT3's valuation.

The target company should not be included in the peer median used to value itself. Otherwise, the analysis would partially use GMAT3's own market multiple to justify GMAT3's valuation, which weakens the purpose of a peer benchmark.


In [3]:
target_ticker = "GMAT3.SA"

target_df = valuation_df[valuation_df["ticker"] == target_ticker].copy()
peers_df = valuation_df[valuation_df["ticker"] != target_ticker].copy()

display(target_df)
display(peers_df)


,ticker,company,sector,subsector,period,report_date,currency,revenue,ebitda,ebit,net_income,cash,total_debt,shares_outstanding,valuation_date,current_price,market_cap,market_cap_mn,net_debt,enterprise_value,ebitda_annualized,ev_ebitda_annualized
0,GMAT3.SA,Grupo Mateus,Consumer Staples,Food Retail,1T26,2026-03-31,BRL,9402,400,NaN,213,1984,2720,"2,300,047,621.00",2026-05-28,4.26,9798203392,"9,798.20",736,"10,534.20",1600,6.58


,ticker,company,sector,subsector,period,report_date,currency,revenue,ebitda,ebit,net_income,cash,total_debt,shares_outstanding,valuation_date,current_price,market_cap,market_cap_mn,net_debt,enterprise_value,ebitda_annualized,ev_ebitda_annualized
1,ASAI3.SA,Assai,Consumer Staples,Cash & Carry,1T26,2026-03-31,BRL,20600,1000,NaN,174,4366,16374,"1,353,531,000.00",2026-05-28,9.25,12411186176,"12,411.19",12008,"24,419.19",4000,6.10
2,PCAR3.SA,GPA,Consumer Staples,Food Retail,1T26,2026-03-31,BRL,4374,458,NaN,-1347,943,4173,NaN,2026-05-28,1.99,978954240,978.95,3230,"4,208.95",1832,2.30


## 3. Peer Group Quality Check

Before calculating valuation, an analyst must assess whether the peer group is clean. A statistical output is only useful if the inputs are economically comparable.

Assai is the cleaner operational comparable because it is a large listed Brazilian food retail and cash-and-carry operator. GPA is also relevant to the sector, but it has a distressed / turnaround profile and may distort peer statistics.

For that reason, this notebook calculates multiple scenarios instead of blindly treating every peer as equally representative.


In [4]:
peers_df["peer_quality"] = np.select(
    [
        peers_df["ticker"].eq("ASAI3.SA"),
        peers_df["ticker"].eq("PCAR3.SA"),
    ],
    [
        "Core Peer",
        "Distressed / Turnaround Peer",
    ],
    default="Review Required",
)

peers_df[
    [
        "ticker",
        "company",
        "peer_quality",
        "ev_ebitda_annualized",
        "net_income",
        "net_debt",
    ]
]


,ticker,company,peer_quality,ev_ebitda_annualized,net_income,net_debt
1,ASAI3.SA,Assai,Core Peer,6.10,174,12008
2,PCAR3.SA,GPA,Distressed / Turnaround Peer,2.30,-1347,3230


## 4. Peer Multiple Statistics

The main valuation multiple is EV/EBITDA. It compares the market value of the whole operating business with annualized EBITDA.

The notebook calculates peer statistics for reference, but the peer group is very small. With only two peers, mean and median should be interpreted as directional valuation anchors, not precise estimates of fair value.


In [5]:
peer_multiple_series = peers_df["ev_ebitda_annualized"].dropna()

peer_stats_df = pd.DataFrame(
    {
        "metric": ["EV/EBITDA Annualized"],
        "mean": [peer_multiple_series.mean()],
        "median": [peer_multiple_series.median()],
        "min": [peer_multiple_series.min()],
        "max": [peer_multiple_series.max()],
        "count": [peer_multiple_series.count()],
    }
)

peer_stats_df


,metric,mean,median,min,max,count
0,EV/EBITDA Annualized,4.20,4.20,2.30,6.10,2


## 5. Scenario Design

Because GPA is distressed, the valuation will use multiple scenarios rather than one mechanical peer average.

The Core Peer Case uses only Assai. The Full Peer Set Case uses Assai and GPA. The Conservative Case uses the lower of the core peer multiple and full peer median. The Upside Case uses a higher reference point based on GMAT3's current trading multiple if available.

These scenarios should not be interpreted with false precision. They are a first-pass range to organize valuation thinking.


In [6]:
core_peer_multiple = peers_df.loc[
    peers_df["ticker"].eq("ASAI3.SA"), "ev_ebitda_annualized"
].iloc[0]

full_peer_median = peers_df["ev_ebitda_annualized"].median()
conservative_multiple = min(core_peer_multiple, full_peer_median)

target_current_ev_ebitda = target_df["ev_ebitda_annualized"].iloc[0]

if pd.notna(target_current_ev_ebitda):
    upside_multiple = max(core_peer_multiple, target_current_ev_ebitda)
else:
    upside_multiple = core_peer_multiple * 1.10

scenario_multiples = {
    "Conservative Case": conservative_multiple,
    "Full Peer Median": full_peer_median,
    "Core Peer Case": core_peer_multiple,
    "Upside Case": upside_multiple,
}

scenario_multiples_df = pd.DataFrame(
    scenario_multiples.items(),
    columns=["scenario", "applied_ev_ebitda_multiple"],
)

scenario_multiples_df


,scenario,applied_ev_ebitda_multiple
0,Conservative Case,4.20
1,Full Peer Median,4.20
2,Core Peer Case,6.10
3,Upside Case,6.58


## 6. Implied Enterprise Value

The applied peer multiple is multiplied by GMAT3 annualized EBITDA to estimate implied enterprise value.

Formula:

`Implied EV = GMAT3 EBITDA annualized × applied multiple`

Because EBITDA is in BRL millions, the implied enterprise value is also in BRL millions.


In [7]:
target_ebitda_annualized = target_df["ebitda_annualized"].iloc[0]
target_net_debt = target_df["net_debt"].iloc[0]
target_shares_outstanding = target_df["shares_outstanding"].iloc[0]
target_current_price = target_df["current_price"].iloc[0]
target_current_market_cap_mn = target_df["market_cap_mn"].iloc[0]
target_current_ev = target_df["enterprise_value"].iloc[0]
target_current_ev_ebitda = target_df["ev_ebitda_annualized"].iloc[0]

valuation_scenarios_df = scenario_multiples_df.copy()
valuation_scenarios_df["implied_enterprise_value"] = (
    target_ebitda_annualized
    * valuation_scenarios_df["applied_ev_ebitda_multiple"]
)

valuation_scenarios_df


,scenario,applied_ev_ebitda_multiple,implied_enterprise_value
0,Conservative Case,4.20,"6,721.81"
1,Full Peer Median,4.20,"6,721.81"
2,Core Peer Case,6.10,"9,767.67"
3,Upside Case,6.58,"10,534.20"


## 7. Equity Value Bridge

Enterprise Value represents the value of the whole business. To get equity value, subtract net debt.

Formula:

`Equity Value = Enterprise Value - Net Debt`

Then:

`Implied Price per Share = Equity Value / Shares Outstanding`

Because EV and net debt are in BRL millions and shares are absolute, equity value must be multiplied by 1,000,000 before dividing by shares.


In [8]:
valuation_scenarios_df["implied_equity_value"] = (
    valuation_scenarios_df["implied_enterprise_value"] - target_net_debt
)

valuation_scenarios_df["implied_price_per_share"] = (
    valuation_scenarios_df["implied_equity_value"] * 1_000_000
) / target_shares_outstanding

valuation_scenarios_df["upside_downside_pct"] = (
    valuation_scenarios_df["implied_price_per_share"] / target_current_price - 1
) * 100

valuation_scenarios_df[
    [
        "scenario",
        "applied_ev_ebitda_multiple",
        "implied_enterprise_value",
        "implied_equity_value",
        "implied_price_per_share",
        "upside_downside_pct",
    ]
]


,scenario,applied_ev_ebitda_multiple,implied_enterprise_value,implied_equity_value,implied_price_per_share,upside_downside_pct
0,Conservative Case,4.20,"6,721.81","5,985.81",2.60,-38.91
1,Full Peer Median,4.20,"6,721.81","5,985.81",2.60,-38.91
2,Core Peer Case,6.10,"9,767.67","9,031.67",3.93,-7.82
3,Upside Case,6.58,"10,534.20","9,798.20",4.26,0.00


## 8. Current Market Positioning

This section compares GMAT3's current trading multiple with peer-implied references. The purpose is to understand whether the market already prices GMAT3 close to its peers, or whether the first-pass comparable set suggests a valuation gap.

This is a positioning exercise, not a final conclusion.


In [9]:
current_positioning_df = pd.DataFrame(
    {
        "reference": [
            "GMAT3 Current EV/EBITDA",
            "ASAI3 EV/EBITDA",
            "PCAR3 EV/EBITDA",
            "Full Peer Median",
        ],
        "ev_ebitda_annualized": [
            target_current_ev_ebitda,
            peers_df.loc[peers_df["ticker"].eq("ASAI3.SA"), "ev_ebitda_annualized"].iloc[0],
            peers_df.loc[peers_df["ticker"].eq("PCAR3.SA"), "ev_ebitda_annualized"].iloc[0],
            full_peer_median,
        ],
    }
)

current_positioning_df


,reference,ev_ebitda_annualized
0,GMAT3 Current EV/EBITDA,6.58
1,ASAI3 EV/EBITDA,6.10
2,PCAR3 EV/EBITDA,2.30
3,Full Peer Median,4.20


## 9. Formatted Valuation Table

This table is only for reading and presentation. It must not replace the numeric `valuation_scenarios_df`, which remains the source for exports, downstream analysis, and charts.


In [10]:
display_df = valuation_scenarios_df.copy()

display_df["applied_ev_ebitda_multiple"] = display_df[
    "applied_ev_ebitda_multiple"
].apply(lambda value: "-" if pd.isna(value) else f"{value:,.1f}x")

display_df["implied_enterprise_value"] = display_df[
    "implied_enterprise_value"
].apply(lambda value: "-" if pd.isna(value) else f"R$ {value:,.0f} mi")

display_df["implied_equity_value"] = display_df[
    "implied_equity_value"
].apply(lambda value: "-" if pd.isna(value) else f"R$ {value:,.0f} mi")

display_df["implied_price_per_share"] = display_df[
    "implied_price_per_share"
].apply(lambda value: "-" if pd.isna(value) else f"R$ {value:,.2f}")

display_df["upside_downside_pct"] = display_df[
    "upside_downside_pct"
].apply(lambda value: "-" if pd.isna(value) else f"{value:,.1f}%")

display_df


,scenario,applied_ev_ebitda_multiple,implied_enterprise_value,implied_equity_value,implied_price_per_share,upside_downside_pct
0,Conservative Case,4.2x,"R$ 6,722 mi","R$ 5,986 mi",R$ 2.60,-38.9%
1,Full Peer Median,4.2x,"R$ 6,722 mi","R$ 5,986 mi",R$ 2.60,-38.9%
2,Core Peer Case,6.1x,"R$ 9,768 mi","R$ 9,032 mi",R$ 3.93,-7.8%
3,Upside Case,6.6x,"R$ 10,534 mi","R$ 9,798 mi",R$ 4.26,0.0%


## 10. Initial Interpretation

GMAT3 currently trades close to Assai on EV/EBITDA, which suggests that the market is already valuing Grupo Mateus broadly in line with the cleaner listed Brazilian comparable in this limited peer set.

GPA produces a much lower EV/EBITDA multiple due to its distressed / turnaround characteristics. Its negative net income and more complex operating situation make it a less clean benchmark for GMAT3, even though it remains relevant as a listed food retail peer.

Including GPA mechanically lowers the peer median and may understate GMAT3 fair value if the objective is to benchmark against normalized, healthier operators. The Assai-only case is probably the cleaner operational benchmark, but relying on one peer is fragile and should not be treated as a complete valuation framework.

This output is a valuation range, not a final target price. A more rigorous version should use LTM EBITDA, a broader peer set, explicit peer inclusion/exclusion logic, and sensitivity analysis around multiples, EBITDA normalization, and net debt.


In [11]:
outputs_dir = Path("../outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)

valuation_scenarios_path = outputs_dir / "gmat3_relative_valuation_scenarios.csv"
current_positioning_path = outputs_dir / "current_market_positioning.csv"

valuation_scenarios_df.to_csv(valuation_scenarios_path, index=False)
current_positioning_df.to_csv(current_positioning_path, index=False)

print(f"Valuation scenarios exported to: {valuation_scenarios_path}")
print(f"Current positioning exported to: {current_positioning_path}")


Valuation scenarios exported to: ../outputs/gmat3_relative_valuation_scenarios.csv
Current positioning exported to: ../outputs/current_market_positioning.csv


## 11. Next Steps

The next notebook should turn the valuation framework into a more complete research output. It should include visualizations, sensitivity tables, a football field chart, and a structured peer inclusion/exclusion discussion.

The analysis should also consider expanding the peer set to LATAM or international food retail and cash-and-carry companies if domestic comparables remain too limited.

Eventually, the project can move toward a final research summary that separates quantitative outputs from analyst judgment, clearly explains assumptions, and avoids presenting mechanical valuation ranges as investment recommendations.
